# ChurnLab: inspect the data and run SQL
Run the project pipeline first if the database is absent. This notebook is a guided starting point, not a completed analysis. Write your observations in your own words.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
raw = pd.read_csv(ROOT / 'data/raw/customer_churn.csv')
raw.head()

## 1. Understand one row
Use DATA-DICTIONARY.md. Which column is the target? Which columns were excluded, and why?

In [ ]:
print('Rows and columns:', raw.shape)
print('Missing cells:', raw.isna().sum().sum())
print('Repeated full rows:', raw.duplicated().sum())
raw.describe().T

## 2. Inspect the source discrepancy
The charge-band documentation says 0–9. What values occur in this file? How would you handle this with a data owner?

In [ ]:
raw['Charge  Amount'].value_counts().sort_index()

## 3. Query tariff-level churn
COUNT counts customers; SUM(churn) counts churn labels; AVG(churn) is the churn fraction for binary labels.

In [ ]:
with sqlite3.connect(ROOT / 'data/processed/churnlab.sqlite') as conn:
    result = pd.read_sql_query('''
    SELECT tariff_plan, COUNT(*) AS customers, SUM(churn) AS churners,
           ROUND(100.0 * AVG(churn), 2) AS churn_rate_pct
    FROM customers GROUP BY tariff_plan
    ''', conn)
result

## 4. Your contribution
Write a CASE expression for subscription bands: under 12 months, 12–35 months, 36+ months. Return counts and churn rates for each band. Explain the result as an association, not a cause.

In [ ]:
# Write your SQL query here, then use pd.read_sql_query as above.


## 5. Protect the evaluation
This should return no rows. Explain why identical profiles across splits would make evaluation less convincing.

In [ ]:
with sqlite3.connect(ROOT / 'data/processed/churnlab.sqlite') as conn:
    overlaps = pd.read_sql_query('''
    SELECT feature_group, COUNT(DISTINCT split) AS partitions
    FROM customers GROUP BY feature_group
    HAVING COUNT(DISTINCT split) > 1
    ''', conn)
overlaps

## 6. Interpretation
Write three findings, two limitations, and one next investigation. Do not use descriptive exploration of the full dataset to retune against the final test set.